# 11 — VFM deliverable: final model → daily predictions → Excel

**Final VFM model** (= benchmark submission v5, 21.089 / 30), chosen by the ablation (notebooks 05–10), the
time-forward and test-separator checks (15), the official-metric re-evaluation (16) and benchmark tests:
```
oil   = physics: q_oil = J · (P_res(t) − Pwf)
          J from Darcy with OUR OWN build-up k & skin (src/own_well_properties.py) for the 6 wells with a
          shut-in; for the 4 wells without one, J fitted from their tests (25 % toward a field prior)
          P_res(t) = LOCAL linear trend per well (Gaussian time weights, 90 days)
gas   = physics: q_oil · Rs / 1000          Rs = solution GOR from the completion file
          (Pwf > bubble point → produced GOR = Rs; the measured test GOR is noisier — benchmark v2 confirmed)
water = ½ · physics + ½ · Gaussian process  physics water: q_oil · WC(t)/(100 − WC(t)), WC linear in time
          (recent tests weighted more, τ = 120 d); the GP only uses separator-safe inputs
daily rate = flowing rate × on_stream_frac  (fully shut-in day → 0)
```
Accuracy (computed below and saved to `outputs/final_model_metrics.json`):
leave-one-test-out (between tests) and time-forward (forecast 0–3 months), with the official metrics.

Outputs:
- `models/vfm_model.pkl` (+ `.joblib`) — trained model (loaded by the UI)
- `outputs/VFM_predictions.xlsx` — sheet `predictions` = required columns; plus `model_info`, `validation`, `daily_details`
- `outputs/plots/vfm_<well>.png` — daily predictions vs well tests
- `outputs/final_model_metrics.json` — accuracy numbers used by the app


In [ ]:
import sys
import warnings
from pathlib import Path

import pickle

import joblib
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import export
import load
import validation as V
import vfm

OUT = load.ROOT / 'outputs'
(OUT / 'plots').mkdir(parents=True, exist_ok=True)

tests = pd.read_parquet(load.PROCESSED / 'test_features.parquet')
daily = pd.read_parquet(load.PROCESSED / 'daily_features.parquet')

# 1. train on ALL 85 tests and save
model = vfm.train_final_model(tests)
joblib.dump(model, load.ROOT / 'models' / 'vfm_model.joblib')
with open(load.ROOT / 'models' / 'vfm_model.pkl', 'wb') as fh:      # .pkl required by the deliverables
    pickle.dump(model, fh)

# 2. predict every well-day
rates = vfm.predict_rates(model, daily)
predictions = export.vfm_table(daily, rates)
print(predictions.shape)
predictions.head()

## Validation of the final model (leave-one-test-out) — goes into the Excel

In [ ]:
import json

final = vfm.final_method()
loto = V.leave_one_test_out(tests, final)
forward = V.time_forward(tests, final)
overall = V.score(tests, loto)
per_well = V.score_per_well(tests, loto).add_suffix(' MAPE %')

metrics = {'model': model['name'], 'physics_options': vfm.FINAL_PHYSICS_OPTIONS, 'n_tests': len(tests)}
for kind, true, pred in [('loto', tests, loto), ('forward', tests.loc[forward.test_index], forward)]:
    for t in V.TARGETS:
        for name, value in V.official_metrics(true[t].values, pred[t].values).items():
            metrics[f'{kind}_{t.split("_")[0]}_{name}'] = round(float(value), 4)
with open(OUT / 'final_model_metrics.json', 'w') as fh:
    json.dump(metrics, fh, indent=1)

print(json.dumps(metrics, indent=1))
per_well.round(2)

In [ ]:
# 3. Excel
flowing = daily[daily.dh_pressure_psia.notna()]
phys = vfm.physics_predict(flowing, model['trends']).reindex(daily.index)
details = pd.DataFrame({
    'date': daily.date.dt.date, 'well_name': daily.well_name,
    'on_stream_frac': daily.on_stream_frac.round(3),
    'no_sensor_data_interpolated': daily.no_data,
    'reservoir_pressure_psia_at_gauge': phys.P_res.round(1),
    'downhole_pressure_psia': daily.dh_pressure_psia.round(1),
    'choke_opening_pct': daily.choke_opening_pct.round(1),
})

model_info = [
    ('model', 'Final VFM: physics + Gaussian-process hybrid (benchmark submission v5, 21.089 / 30)'),
    ('oil', 'physics q_oil = J * (P_res(t) - Pwf); J from Darcy with our own build-up k & skin (6 wells) or fitted from tests (4 wells without a shut-in); P_res(t) local linear trend per well (Gaussian time weights, 90 days)'),
    ('gas', 'q_gas = q_oil * Rs / 1000; Rs = solution GOR from the completion file (formation gas, excludes lift gas)'),
    ('water', '0.5 * physics (q_oil * WC(t)/(100-WC(t)), WC linear in time, recent tests weighted, tau 120 d) + 0.5 * Gaussian process on separator-safe inputs'),
    ('test-separator effect', 'tests run at lower flowline pressure than normal days; physics handles it via Pwf, GP uses only separator-safe inputs'),
    ('daily rate', 'flowing rate * fraction of the day on stream; fully shut-in day = 0'),
    ('training data', '85 well tests, cleaned (5 sign flips, GL-06 gas Sm3/d -> Mscf/d)'),
    ('validation', 'leave-one-test-out (between tests) and time-forward (train before cutoff, predict next 0-3 months)'),
    ('LOTO WAPE oil / water / gas %', f"{metrics['loto_oil_WAPE']:.2f} / {metrics['loto_water_WAPE']:.2f} / {metrics['loto_gas_WAPE']:.2f}"),
    ('forecast WAPE oil / water / gas %', f"{metrics['forward_oil_WAPE']:.2f} / {metrics['forward_water_WAPE']:.2f} / {metrics['forward_gas_WAPE']:.2f}"),
    ('note on gas validation', 'gas is validated against measured test GOR, which is noisier than the true GOR (= Rs); the benchmark improved with Rs'),
    ('units', 'oil, water: STB/d; gas: Mscf/d'),
]
validation_sheet = pd.concat([overall.round(3), per_well.round(2)])

export.write_vfm_excel(OUT / 'VFM_predictions.xlsx', predictions, model_info, validation_sheet, details)
print('written:', OUT / 'VFM_predictions.xlsx')

## 4. Daily predictions vs well tests (dots) — grey bands = shut-in days

In [ ]:
def plot_vfm_well(well, save=True):
    d = predictions[predictions.well_name == well]
    t = tests[tests.well_name == well]
    shut = daily[(daily.well_name == well) & (daily.on_stream_frac < 1)].date
    fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
    for ax, col, color, unit in zip(axes, ['oil_rate_stbd', 'water_rate_stbd', 'gas_rate_mscfd'],
                                    ['tab:green', 'tab:blue', 'tab:red'], ['STB/d', 'STB/d', 'Mscf/d']):
        ax.plot(pd.to_datetime(d.date), d[col], color=color, lw=1, label='VFM daily prediction')
        ax.plot(t.time, t[col], 'ko', ms=5, label='well test (measured)')
        for day in shut:
            ax.axvspan(day, day + pd.Timedelta(days=1), color='grey', alpha=0.25)
        ax.set_ylabel(f'{col.split("_")[0]} [{unit}]')
    axes[0].legend(loc='upper right', fontsize=8)
    fig.suptitle(f'{well}: VFM daily rates')
    fig.tight_layout()
    if save:
        fig.savefig(OUT / 'plots' / f'vfm_{well}.png', dpi=110)
    plt.show()

for well in sorted(daily.well_name.unique()):
    plot_vfm_well(well)